# Are lunar terrain gaps associated with shadow, or introduced by processing?

**Historical investigation, not current terrain coverage.** The original Earth tile/halo defect was corrected in [notebook 09](09_verify_earth_coverage_fix.ipynb). Lunar source heights were finite; our image-support mask introduced the elevation holes. [Notebook 13](13_build_lunar_terrain_views.ipynb) supplies a full-context view with zero elevation gaps inside the seven ROIs. Read [notebook 14](../exploration/14_terrain_feature_eda.ipynb) for the corrected terrain and separate support views. Remaining derivative edge exclusions and genuine environmental missingness are different issues.

This read-only investigation compares published full elevation, maximum-lit imagery, image counts, our support-filtered elevation, and native 60 m illumination on the existing 30 m analysis grid. It verifies the documented elevation and derivative validity rules across all seven regions.

**Interpretation:** notebook 10 grayscale elevation/slope maps do not depict sunlight. Here actual image brightness and modeled visibility provide separate evidence. Missingness can follow real low illumination because our policy excludes weakly image-supported terrain. That does not imply the published height is absent or corrupt.

Run All in the project environment. This investigation refreshes its own fixed diagnostic folder `notebooks/diagnostics/shadow-gap-investigation-20261028/`; raw/processed data are read-only. No external code is run. See `EXTERNAL_RESEARCH.md` there for producer code, NASA documentation, and GitHub examples. Statistical comparisons are descriptive, spatially correlated, and not causal tests. Orthomosaic averages are only brightness proxies, not an attempt to reconstruct the producer's exact image threshold.

**Workflow stage: preprocessing / data readiness.** Use this notebook to inspect coverage, investigate exclusions, or verify corrections before EDA. Its saved results and pinned inputs preserve the original investigation. Notebook 13 builds the two lunar terrain views; notebook 14 explores the corrected Earth run and those views. See the [workflow and acceptance checks](../README.md). Moving this notebook does not change when the original findings were made.


In [ ]:
"""Read-only spatial evidence for the lunar missing-data question; no threshold changes."""
import json
import sys
from pathlib import Path

import numpy as np
import rasterio
import scipy
from rasterio.enums import Resampling
from rasterio.warp import reproject
from scipy.ndimage import minimum_filter

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from inspection import check_input, file_hash, verify_run

RUN = ROOT / 'data/processed/lunar_terrain_environment/20261028T022855014566Z_cffea7'
OUT = ROOT / 'notebooks/diagnostics/shadow-gap-investigation-20261028'
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'local_provenance.json').write_text(json.dumps({'status': 'running'}), encoding='utf-8')
expected = verify_run(RUN)
evidence = {}
rows = []
gallery = ['<!doctype html><meta charset="utf-8"><title>Lunar source and mask comparison</title>',
           ('<h1>Lunar source and mask comparison</h1><p>Read-only diagnostic. '
           'All panels share the region output grid. Gray DEM values are elevation, not illumination. '
           'Magenta is missing. Panels are decimated previews; statistics use all 30 m cells. '
           'Solar and shadow values are sampled from 60 m native crops, not independent 30 m observations.</p>')]


def read(path):
    check_input(path, expected, evidence)
    with rasterio.open(path) as src:
        return src.read(1, masked=True).astype('float32').filled(np.nan)


def aligned(path, grid, method=Resampling.nearest):
    """Decode once and align a diagnostic view; never write to the source."""
    evidence[str(path)] = file_hash(path)
    shape, transform, crs = grid
    result = np.full(shape, np.nan, dtype='float32')
    with rasterio.open(path) as src:
        assert src.crs is not None
        reproject(rasterio.band(src, 1), result, src_transform=src.transform, src_crs=src.crs,
                  src_nodata=src.nodata, dst_transform=transform, dst_crs=crs,
                  dst_nodata=np.nan, resampling=method)
        result = result * src.scales[0] + src.offsets[0]
    return result


def preview(values, name, lo, hi):
    stride = max(1, int(np.ceil(max(values.shape) / 600)))
    sample = values[::stride, ::stride]
    intensity = (255 * np.clip(np.nan_to_num((sample - lo) / max(hi - lo, 1e-12)), 0, 1)).astype('uint8')
    rgb = np.repeat(intensity[:, :, None], 3, axis=2)
    rgb[~np.isfinite(sample)] = [220, 35, 160]
    with rasterio.open(OUT / name, 'w', driver='PNG', width=rgb.shape[1], height=rgb.shape[0],
                       count=3, dtype='uint8') as dst:
        dst.write(rgb.transpose(2, 0, 1))




## Check all regions and save aligned comparison panels

Elevation exclusions must exactly match `image_support_fraction < 0.99` inside the ROI. Derived feature validity must match the 3?3 or 11?11 complete-elevation neighborhood. A mismatch stops the notebook. This tests consistency with our policy, not whether that policy is optimal or every source is perfectly registered. Source imagery/elevation are averaged to 30 m for comparison; counts and native illumination use nearest sampling. The raw source arrays remain unchanged.

In [ ]:
folders = sorted((RUN / 'moon').glob('*/grid_30m'))
expected_regions = {
    'A3CLR22_10_Leibnitz_Beta_Plateau', 'A3CLR22_11_Malapert_Massif',
    'A3CLR22_12_deGerlache_Kocher_Massif', 'A3CLR22_2_Haworth',
    'A3CLR22_3_Faustini_Rim_A', 'A3CLR22_4_deGerlache_Rim_2',
    'A3CLR22_5_Connecting_Ridge_Extension',
}
assert {p.parent.name for p in folders} == expected_regions
for folder in folders:
    region = folder.parent.name
    source = ROOT / 'data/raw/moon/sdem/regions' / region
    elevation = read(folder / 'elevation_m.tif')
    fraction = read(folder / 'image_support_fraction.tif')
    with rasterio.open(folder / 'elevation_m.tif') as src:
        grid = src.shape, src.transform, src.crs
    inside = np.isfinite(fraction)
    missing = inside & ~np.isfinite(elevation)
    mismatch = int(np.count_nonzero(missing ^ (inside & (fraction < .99))))
    assert mismatch == 0, region
    neighborhood = {}
    finite = np.isfinite(elevation)
    for feature, window in [('slope_deg', 3), ('laplacian_per_m', 3), ('local_elevation_std_m', 11)]:
        actual = np.isfinite(read(folder / (feature + '.tif')))
        predicted = minimum_filter(finite.astype('uint8'), size=window, mode='constant', cval=0).astype(bool)
        neighborhood[feature] = int(np.count_nonzero(actual ^ predicted))
        assert neighborhood[feature] == 0, (region, feature)
    raw = aligned(next(source.glob('*_GLDELEV_005.tif')), grid, Resampling.average)
    ortho = aligned(next(source.glob('*_GLDOMOS_005.tif')), grid, Resampling.average)
    counts = aligned(next(source.glob('*_GLDMASK_005.tif')), grid)
    native = folder.parent / 'native/illumination'
    solar_path = native / 'AVGVISIB_85S_060M_201608.TIF'
    shadow_path = native / 'LPSR_85S_060M_201608.TIF'
    check_input(solar_path, expected, evidence)
    check_input(shadow_path, expected, evidence)
    solar = aligned(solar_path, grid)
    shadow = aligned(shadow_path, grid)
    conditions = {}
    for label, subset in [('missing_elevation', missing), ('retained_elevation', inside & ~missing)]:
        valid_solar = subset & np.isfinite(solar)
        valid_ortho = subset & np.isfinite(ortho)
        valid_shadow = subset & np.isfinite(shadow)
        conditions[label] = {
            'cells': int(subset.sum()), 'solar_valid_cells': int(valid_solar.sum()),
            'median_solar_visibility': float(np.median(solar[valid_solar])) if valid_solar.any() else None,
            'orthomosaic_valid_cells': int(valid_ortho.sum()),
            'median_orthomosaic_value': float(np.median(ortho[valid_ortho])) if valid_ortho.any() else None,
            'shadow_valid_cells': int(valid_shadow.sum()),
            'permanent_shadow_fraction': float(np.mean(shadow[valid_shadow] == 1)) if valid_shadow.any() else None,
        }
    row = {'region': region, 'roi_cells': int(inside.sum()),
           'raw_averaged_elevation_missing_inside': int((inside & ~np.isfinite(raw)).sum()),
           'processed_elevation_missing_inside': int(missing.sum()),
           'elevation_rule_mismatches': mismatch, 'neighborhood_rule_mismatches': neighborhood,
           'support_fractions': {key: int((inside & condition).sum()) for key, condition in {
               'zero': fraction == 0, 'between_zero_and_099': (fraction > 0) & (fraction < .99),
               'at_least_099': fraction >= .99}.items()}, 'conditional_statistics': conditions}
    rows.append(row)
    lo, hi = np.nanquantile(raw[inside], [.01, .99])
    gallery.append(f'<h2>{region}</h2><div style="display:flex;flex-wrap:wrap">')
    panels = [('raw_elevation', raw, lo, hi, 'Full published elevation, averaged to 30 m'),
              ('processed_elevation', elevation, lo, hi, 'Our image-supported elevation; same display range'),
              ('orthomosaic', ortho, 0, float(np.nanquantile(ortho, .99)), 'Maximum-lit image mosaic: black = low image value'),
              ('image_counts', counts, 0, float(np.nanquantile(counts, .99)), 'Contributing-image count: black = zero'),
              ('solar_visibility', solar, 0, 1, 'Solar visibility fraction: black = 0, white = 1'),
              ('permanent_shadow', shadow, 0, 1, 'Permanent shadow: white = shadow, black = not shadow')]
    for key, values, low, high, caption in panels:
        filename = f'{region}_{key}.png'
        preview(values, filename, low, high)
        gallery.append(f'<figure><figcaption>{caption}<br>Display {low:.4g} to {high:.4g}</figcaption>'
                       f'<img width="400" src="{filename}" alt="{caption}"></figure>')
    gallery.append('</div>')
    print(json.dumps(row), flush=True)

(OUT / 'local_checks.json').write_text(json.dumps(rows, indent=2), encoding='utf-8')
(OUT / 'gallery.html').write_text('\n'.join(gallery), encoding='utf-8')
notebook = ROOT / 'notebooks/preprocessing/12_investigate_lunar_shadow_gaps.ipynb'
evidence[str(notebook)] = file_hash(notebook)
evidence[str(ROOT / 'notebooks/inspection.py')] = file_hash(ROOT / 'notebooks/inspection.py')
(OUT / 'local_provenance.json').write_text(json.dumps({'status': 'checks_complete', 'inputs_sha256': evidence,
    'comparison': 'nearest native illumination and mask; mean raw elevation and orthomosaic to existing 30m grid',
    'limitations': 'association, not causation; no subpixel registration validation; inherited LOLA prior is not independent SfS detail',
    'versions': {'numpy': np.__version__, 'rasterio': rasterio.__version__, 'scipy': scipy.__version__, 'python': sys.version}}, indent=2), encoding='utf-8')


## Read the evidence

The full gallery has six panels per region, including raw and processed elevations with identical display ranges. These PNG previews are not georeferenced rasters. Solar visibility and permanent shadow are sampled from 60 m products; they do not become independent 30 m observations. Permanent shadow is only one reason for weak image support, so not every excluded pixel should be classified as permanent shadow.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

summary = pd.DataFrame([{
    'region': r['region'],
    'raw_30m_missing': r['raw_averaged_elevation_missing_inside'],
    'excluded_elevation_cells': r['processed_elevation_missing_inside'],
    'median_solar_missing_pct': 100*r['conditional_statistics']['missing_elevation']['median_solar_visibility'],
    'median_solar_retained_pct': 100*r['conditional_statistics']['retained_elevation']['median_solar_visibility'],
} for r in rows])
display(summary)

for key in ['raw_elevation', 'processed_elevation', 'orthomosaic', 'image_counts']:
    display(Markdown(key.replace('_', ' ')))
    display(Image(filename=str(OUT / f'A3CLR22_3_Faustini_Rim_A_{key}.png'), width=450))
print('Full comparison gallery:', OUT / 'gallery.html')
print('Scientific and external-source interpretation:', OUT / 'REPORT.md')

manifest_path = OUT / 'local_provenance.json'
manifest = json.loads(manifest_path.read_text())
manifest['status'] = 'complete'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')


**Faustini example: raw elevation, processed elevation, maximum-lit imagery, image count.**